# 深度學習框架應用（大學部）｜主題 7：模型訓練技巧

這本筆記本配合投影片《主題 7：模型訓練技巧》。主角是 Fashion-MNIST（10 類衣服鞋子的 28×28 灰階照片），我們只拿 10,000 張訓練、2,000 張驗證，CPU 也跑得完。五題依序練：深網路的梯度與初始化 → BatchNorm → 最佳化器 → 學習率排程 → 正則化與早停。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 深網路的梯度量一量（初始化） | 第 121 頁 | 第 15～20 頁 |
| 二 | 加 BatchNorm 前後 | 第 122 頁 | 第 24～28 頁 |
| 三 | 三種最佳化器賽跑 | 第 123 頁 | 第 35～47 頁 |
| 四 | 畫出四種學習率排程 | 第 124 頁 | 第 59～69 頁 |
| 五 | Dropout＋權重衰減＋早停 | 第 125 頁 | 第 77～89 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W07_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 由上往下跑：先跑「環境準備」「準備資料」「共用小工具」三格，之後每一題如果有「準備（直接執行）」那一格，也要先跑。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 有 GPU 會自動用（「執行階段 › 變更執行階段類型」），沒有也沒關係：每一題在 CPU 上幾十秒到兩分鐘就跑完。在 GPU 上跑，數字的最後幾位會和投影片不一樣，大小與趨勢相同就對了。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

下面三格五題都會用到：載入套件、準備 Fashion-MNIST、幾個共用小工具（`loader` 裝盒、`evaluate` 評估、`train_epoch` 練一輪、`show` 畫圖）。第一次跑會下載資料，大約 30 秒。

In [ ]:
# ===== 環境準備（直接執行）=====
import copy, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch：', torch.__version__, '｜裝置：', dev)

In [ ]:
# ===== 準備資料：Fashion-MNIST（直接執行，不用改）=====
raw = datasets.FashionMNIST('data', train=True, download=True)
X = (raw.data.float() / 255 - 0.2860) / 0.3530    # 標準化：平均 0、標準差 1
y = raw.targets                                     # 0～9：T恤、褲子…短靴
g = torch.Generator().manual_seed(42)              # 固定亂數，大家切到同一批
idx = torch.randperm(len(X), generator=g)
Xtr, ytr = X[idx[:10000]], y[idx[:10000]]          # 訓練用 10,000 張
Xva, yva = X[idx[10000:12000]], y[idx[10000:12000]]  # 驗證用 2,000 張
print('訓練：', tuple(Xtr.shape), '驗證：', tuple(Xva.shape))

In [ ]:
# ===== 共用小工具（直接執行，不用改）=====
def loader(n=10000, bs=128):
    """取前 n 張訓練資料，每盒 bs 張；每一輪洗牌，洗法固定"""
    ds = TensorDataset(Xtr[:n], ytr[:n])
    return DataLoader(ds, batch_size=bs, shuffle=True,
                      generator=torch.Generator().manual_seed(0))

def evaluate(model, X_, y_):
    """回傳 (loss, 準確率)。評估前一定先切 eval()"""
    model.eval()
    with torch.no_grad():
        out = model(X_.to(dev))
        loss = F.cross_entropy(out, y_.to(dev)).item()
        acc = (out.argmax(1) == y_.to(dev)).float().mean().item()
    return loss, acc

def train_epoch(model, opt, dl=None):
    """練一個 epoch，回傳平均訓練 loss；沒給 dl 就用 loader()。訓練前先切 train()"""
    if dl is None: dl = loader()
    model.train(); total = 0.0
    for xb, yb in dl:
        xb, yb = xb.to(dev), yb.to(dev)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(xb)
    return total / len(dl.dataset)

def r3(xs):
    """一串數字都四捨五入到小數第 3 位，印出來比較好讀"""
    return [round(v, 3) for v in xs]

def show(curves, ylabel, xlabel='epoch', logy=False):
    """把好幾條曲線畫在同一張圖（圖上的字用英文，Colab 才不會變方塊）"""
    for name, ys in curves.items():
        plt.plot(range(1, len(ys) + 1), ys, 'o-', ms=3, label=name)
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    if logy: plt.yscale('log')
    plt.grid(alpha=.3); plt.legend(); plt.show()

## 練習一：深網路的梯度量一量（初始化） (20%)

> 題目在投影片**第 121 頁**｜觀念在第 15～20 頁

**情境**：小智疊了一座 10 層的深網路（每層 100 個神經元），想知道誤差從最後一層往回傳的時候，每一層分到的梯度有多大。就像傳話遊戲：話傳到第一個人時，還聽不聽得見？

**這題在練**：親眼看到梯度消失和梯度爆炸，再用 He 初始化（Kaiming initialization）把它救回來

**步驟**

1. 先執行「練習一：準備」：`deep_mlp(act, init)` 會疊出 10 層網路，`grad_sizes` 會量出每一層權重梯度的大小
2. 在 ✏️ 格填好第三組設定（ReLU＋He 常態初始化），以及反向傳播那一行
3. 比一比三組的第 1 層和第 10 層：哪一組消失、哪一組爆炸、哪一組剛好

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習一：準備（直接執行，不用改）=====
def deep_mlp(act, init=None, depth=10, width=100):
    """疊 10 層隱藏層；init 是初始化函式，None 就用 PyTorch 預設"""
    torch.manual_seed(42)                    # 每次都從同一組亂數開始
    layers = [nn.Flatten()]
    for i in range(depth):
        lin = nn.Linear(784 if i == 0 else width, width)
        if init: init(lin.weight)
        nn.init.zeros_(lin.bias)             # 偏差一律從 0 開始
        layers += [lin, act()]
    return nn.Sequential(*layers, nn.Linear(width, 10))

def grad_sizes(model):
    """每一層 Linear 權重梯度的大小（向量長度），第 1 層排最前面"""
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]

xb, yb = Xtr[:256], ytr[:256]                # 固定拿前 256 張來量

In [ ]:
# ✏️ 練習一：三種設定，量一量每一層的梯度
he = nn.init.___                       # ✏️ He 常態初始化
setups = [('sigmoid + default', nn.Sigmoid, None),
          ('ReLU + N(0,1)', nn.ReLU, nn.init.normal_),
          ('ReLU + He', nn.___, he)]   # ✏️ ReLU
grads = {}
for name, act, init in setups:
    model = deep_mlp(act, init)        # 準備格寫好的
    loss = F.cross_entropy(model(xb), yb)
    loss.___()                         # ✏️ 反向傳播
    grads[name] = g = grad_sizes(model)
    print(f'{name:18} 第1層 {g[0]:.1e} 第10層 {g[9]:.1e}')
show(grads, 'grad size', xlabel='layer', logy=True)

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行「第1層、第10層」的梯度大小，以及對數座標的折線圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 「sigmoid + default」那一組，第 1 層和第 10 層的梯度差了幾個數量級？這叫什麼問題？為什麼 sigmoid 容易造成它？**

（在這裡寫）

**2. 「ReLU + N(0,1)」那一組發生什麼事？He 初始化把權重的變異數設成多少？第一層（784 個輸入）的標準差大約是多少？**

（在這裡寫）

## 練習二：加 BatchNorm 前後 (20%)

> 題目在投影片**第 122 頁**｜觀念在第 24～28 頁

**情境**：同一座 6 層 sigmoid 的深網路、同一個最佳化器（SGD，學習率 0.1）、同樣練 5 個 epoch，唯一的差別是每個 Linear 後面有沒有插一層 BatchNorm（BN）。

**這題在練**：看到 BatchNorm 對深網路有多大幫助，並認識 BN 層裡的參數和緩衝

**步驟**

1. 先執行「練習二：準備」：`sig_net(bn_layer)` 會疊出 6 層 sigmoid 網路，給了 `bn_layer` 就在每個 Linear 後面插 BN
2. 在 ✏️ 格填好 BN 的類別名稱，跑沒有 BN 和有 BN 各 5 個 epoch，印出每個 epoch 的驗證準確率
3. 找出第一個 BN 層在 `model` 的第幾格（從 0 數：Flatten、Linear、BN…），印出它的參數和緩衝名字

> 這一題要訓練兩次，CPU 上大約 1 分鐘。

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行，不用改）=====
def sig_net(bn_layer=None, depth=6, width=100):
    """6 層 sigmoid 的深網路；給了 bn_layer，就在每個 Linear 後面插一層"""
    torch.manual_seed(42)
    layers = [nn.Flatten()]
    for i in range(depth):
        layers.append(nn.Linear(784 if i == 0 else width, width))
        if bn_layer: layers.append(bn_layer(width))
        layers.append(nn.Sigmoid())
    return nn.Sequential(*layers, nn.Linear(width, 10)).to(dev)

In [ ]:
# ✏️ 練習二：同一個深網路，加 BatchNorm 前後
bn_layer = nn.___                # ✏️ 一維的 BatchNorm
accs = {'no BN': [], 'BN': []}
for name, acc in accs.items():
    model = sig_net(bn_layer if name == 'BN' else None)
    opt = optim.SGD(model.parameters(), lr=0.1)
    for ep in range(5):
        train_epoch(model, opt)  # 練一個 epoch
        acc += [round(evaluate(model, Xva, yva)[1], 3)]
    print(name, acc)
bn1 = model[___]                 # ✏️ 第一個 BN 在第幾格
print('參數：', [n for n, _ in bn1.named_parameters()])
print('緩衝：', [n for n, _ in bn1.named_buffers()])
show(accs, 'val acc')

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「沒有 BN」「有 BN」兩行準確率、BN 的參數與緩衝名字，以及兩條準確率曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 沒有 BN 的那一組，驗證準確率為什麼一直停在 0.1 左右？和練習一有什麼關係？**

（在這裡寫）

**2. BN 層的兩個參數（weight、bias）和三個緩衝（running_mean、running_var、num_batches_tracked）各在做什麼？為什麼評估前一定要 `model.eval()`？**

（在這裡寫）

## 練習三：三種最佳化器賽跑 (20%)

> 題目在投影片**第 123 頁**｜觀念在第 35～47 頁

**情境**：同一個 MLP（784 → 300 → 100 → 10）、同一組起始權重（seed 42）、同樣的資料順序，只換最佳化器：SGD、SGD＋Momentum、Adam，各練 5 個 epoch。

**這題在練**：一次只改一個變因，比較最佳化器下山的速度

**步驟**

1. 先執行「練習三：準備」：`mlp()` 每次都會從同一組亂數重建模型
2. 在 ✏️ 格填好 Momentum 的動量值，以及 Adam 的類別名稱
3. 比一比三條訓練 loss 曲線：誰下降得最快？第 5 個 epoch 誰最低？

> 這一題要訓練三次，CPU 上大約 1 分鐘。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行，不用改）=====
def mlp():
    """784 → 300 → 100 → 10 的 MLP，每次都從同一組亂數開始"""
    torch.manual_seed(42)
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, 300), nn.ReLU(),
                         nn.Linear(300, 100), nn.ReLU(),
                         nn.Linear(100, 10)).to(dev)

In [ ]:
# ✏️ 練習三：同一個起點，三種最佳化器賽跑
losses = {}
for name in ['SGD', 'Momentum', 'Adam']:
    model = mlp()  # 每次都從 seed 42 重建
    p = model.parameters()
    if name == 'SGD':
        opt = optim.SGD(p, lr=.01)
    elif name == 'Momentum':
        opt = optim.SGD(p, lr=.01, momentum=___)  # ✏️ 0.9
    else:
        opt = optim.___(p, lr=1e-3)  # ✏️ Adam
    ls = [train_epoch(model, opt) for ep in range(5)]
    losses[name] = ls; print(name, r3(ls))
show(losses, 'train loss')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行訓練 loss（SGD、Momentum、Adam）和三條曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 第 1 個 epoch 結束時，三種最佳化器的訓練 loss 各是多少？哪一個最慢？Momentum 為什麼比 SGD 快？**

（在這裡寫）

**2. 為什麼三組都要「從 seed 42 重建模型」？如果 Adam 那一組沿用 Momentum 練完的模型，會怎樣？**

（在這裡寫）

## 練習四：畫出四種學習率排程 (20%)

> 題目在投影片**第 124 頁**｜觀念在第 59～69 頁

**情境**：不用真的訓練，只拿一個假參數的最佳化器，看四種排程器每一步把學習率改成多少：StepLR（每 30 回合 ×0.1）、餘弦（15 回合降到 0）、OneCycle（20 個 epoch × 391 批＝7,820 步）、ReduceLROnPlateau（看 11 回合示範用的 val_acc）。

**這題在練**：認得四種排程器的名字和參數，親手驗證投影片上的數字

**步驟**

1. 先執行「練習四：準備」：`op(lr)` 造一個假的最佳化器，`trace` 記下每一步的學習率，`plateau` 把 11 個 val_acc 依序交給排程
2. 在 ✏️ 格看參數填出三個排程器的類別名稱，以及 OneCycle 的最高學習率
3. 對照投影片：StepLR 第 0／30／60／90 回合、Cosine 第 5、10 回合、OneCycle 的起點和最高點、Plateau 在哪一回合降

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行，不用改）=====
from torch.optim import lr_scheduler as LS

def op(lr):
    """只有一個假參數的 SGD，專門拿來看學習率怎麼變"""
    return torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=lr)

def trace(sched, n):
    """呼叫 n 次 step()，記下每一次開始時的學習率"""
    lrs = []
    for _ in range(n):
        lrs.append(sched.optimizer.param_groups[0]['lr'])
        sched.optimizer.step(); sched.step()
    return lrs

VAL = [.80, .84, .85, .85, .849, .85, .848, .849, .847, .849, .848]  # 示範用

def plateau(sched, vals):
    """每一回合把 val_acc 交給排程，記下這一回合結束後的學習率"""
    lrs = []
    for v in vals:
        sched.step(v)
        lrs.append(sched.optimizer.param_groups[0]['lr'])
    return lrs

def fmt(xs, *ks):
    """把學習率印成好讀的樣子；給了 ks 就只印那幾個位置"""
    return [float(f'{xs[k]:.4g}') for k in (ks or range(len(xs)))]

def report4(st, co, oc, pl):
    print('StepLR 第 0/30/60/90 回合：', fmt(st, 0, 30, 60, 90))
    print('Cosine 第 5、10 回合：', fmt(co, 5, 10))
    print('OneCycle 起點：', fmt(oc, 0), '｜最高點在第', oc.index(max(oc)) + 1, '步：', max(oc))
    print('Plateau 每一回合：', fmt(pl))
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    for a, ys, t, xl in [(ax[0], st, 'StepLR', 'epoch'), (ax[1], co, 'Cosine', 'epoch'),
                         (ax[2], oc, 'OneCycle', 'step (batch)')]:
        a.plot(ys); a.set_title(t); a.set_xlabel(xl); a.grid(alpha=.3)
    ax[0].set_ylabel('learning rate'); plt.tight_layout(); plt.show()

In [ ]:
# ✏️ 練習四：四種排程，學習率怎麼變
s1 = LS.___(op(.1), step_size=30, gamma=.1)  # ✏️ 階梯
s2 = LS.___(op(1e-3), T_max=15)       # ✏️ 餘弦
s3 = LS.OneCycleLR(op(1e-3), max_lr=___,  # ✏️ 最高點
                   total_steps=7820)  # 20×391 步
s4 = LS.___(op(1e-3),                 # ✏️ 看分數
            mode='max', factor=0.5, patience=3)
st = trace(s1, 100)                   # 走 100 回合
co = trace(s2, 15)                    # 走 15 回合
oc = trace(s3, 7820)                  # 走 7,820 步
pl = plateau(s4, VAL)                 # 11 回合示範分數
report4(st, co, oc, pl)               # 印數字、畫圖

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：四行印出的學習率，以及三張學習率曲線圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. StepLR 在第 75 回合的學習率是多少？OneCycle 的 step() 為什麼要放在每個 batch 之後？如果只在每個 epoch 結束呼叫（20 次），學習率會停在哪裡？**

（在這裡寫）

**2. ReduceLROnPlateau 在第幾回合第一次把學習率降成 5e-4？為什麼是那一回合？**

（在這裡寫）

## 練習五：Dropout＋權重衰減＋早停 (20%)

> 題目在投影片**第 125 頁**｜觀念在第 77～89 頁

**情境**：故意只給 1,000 張訓練照片，再配一個很大的 MLP（784 → 512 → 512 → 10），模型很容易把訓練資料背起來。準備格會先練一個「什麼正則化都不加」的對照組 30 回合。

**這題在練**：親眼看到過擬合，再用 Dropout、權重衰減（weight decay）和早停（early stopping）把它管住，並保留最佳那一回合的權重

**步驟**

1. 先執行「練習五：準備＋對照組」，看對照組的訓練 loss 和驗證 loss 各往哪裡走
2. 在 ✏️ 格填好 Dropout 比例、複製權重的函式和 patience，最多練 40 回合
3. 比一比兩條驗證 loss 曲線：誰比較低？在第幾回合停下？

> 對照組 30 回合加上 ✏️ 格，CPU 上大約 1～2 分鐘。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備＋對照組（直接執行，不用改，約 1 分鐘）=====
from torch.optim import AdamW

def big_mlp(p=0.0):
    """784 → 512 → 512 → 10；p > 0 就在每層 ReLU 後面加 Dropout"""
    torch.manual_seed(42)
    layers = [nn.Flatten()]
    for n_in in [784, 512]:
        layers += [nn.Linear(n_in, 512), nn.ReLU()]
        if p > 0: layers.append(nn.Dropout(p))
    return nn.Sequential(*layers, nn.Linear(512, 10)).to(dev)

small = loader(n=1000, bs=64)        # 故意只給 1,000 張，很容易背起來
model = big_mlp()                    # 對照組：不加任何正則化，練滿 30 回合
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
base = {'train': [], 'val': []}
for ep in range(30):
    base['train'].append(train_epoch(model, opt, small))
    base['val'].append(evaluate(model, Xva, yva)[0])
k = base['val'].index(min(base['val']))
print(f"對照組：訓練 loss 最後 {base['train'][-1]:.3f}；"
      f"驗證 loss 第 {k + 1} 回合最低 {base['val'][k]:.3f}，第 30 回合變成 {base['val'][-1]:.3f}")
show(base, 'loss (baseline)')

def one_round(model, opt):
    """用那 1,000 張練一回合，回傳驗證 loss"""
    train_epoch(model, opt, small)
    return evaluate(model, Xva, yva)[0]

def report(model, ep, best, hist):
    acc = evaluate(model, Xva, yva)[1]
    print(f'第 {ep} 回合停下；最佳驗證 loss {best:.3f}（第 {hist.index(best) + 1} 回合），'
          f'載回後驗證準確率 {acc:.3f}')
    show({'baseline': base['val'], 'dropout + wd + early stop': hist}, 'val loss')

In [ ]:
# ✏️ 練習五：Dropout＋權重衰減＋早停
model = big_mlp(p=___)           # ✏️ Dropout 比例 0.3
opt = AdamW(model.parameters(), weight_decay=0.01)
best, best_state, wait, hist = 9e9, None, 0, []
for ep in range(1, 41):
    vl = one_round(model, opt)   # 練一回合，回傳驗證 loss
    hist.append(vl); wait += 1   # 先當作沒進步
    if vl < best - 1e-4:         # 真的進步才算
        best, wait = vl, 0
        sd = model.state_dict()  # 這只是「連結」
        best_state = copy.___(sd)  # ✏️ 另外複製一份
    if wait >= ___: break        # ✏️ 忍 5 回合
model.load_state_dict(best_state)  # 載回最佳那回合
report(model, ep, best, hist)    # 印結果、畫圖

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：對照組那一行、「第幾回合停下」那一行，以及兩條驗證 loss 曲線

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 對照組的訓練 loss 和驗證 loss 各往哪裡走？從第幾回合開始過擬合？**

（在這裡寫）

**2. 為什麼 best_state 要用 `copy.deepcopy`？如果寫成 `best_state = model.state_dict()` 會怎樣？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：BatchNorm 的 running_mean 是什麼） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W07_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。